<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>

<br>

# <font color="#76b900">**Notebook 4:** Running State Chains</font>

<br>

In the previous notebook, we introduced some key LangChain Expression Language (LCEL) material regarding runnables. By now, you should be comfortable with both internal and external reasoning, as well as how to develop pipelines that facilitate it! In this notebook, we will make our way towards more advanced paradigms that will allow us to orchestrate more complex dialog management strategies and begin to execute on longer-form document reasoning.

<br>

### **Learning Objectives:**

- Learning how to leverage runnables to orchestrate interesting LLM systems.  
- Understanding how running state chains can be used for dialog management and iterative decision-making.

<br>

### **Questions To Think About:**

- Would there ever be a use for a single-module variant of the running state chain that is not constantly querying the environment for input?
- You may notice that the JSON prediction is actually working pretty well. It might not always work so well depending on the questions and the JSON format complexity. What kinds of issues do you expect to encounter in this regard?
- What kinds of approaches can you think of completely swapping prompts as part of the running state chain?

<br>

### **Notebook Source:**

- This notebook is part of a larger [**NVIDIA Deep Learning Institute**](https://www.nvidia.com/en-us/training/) course titled [**Building RAG Agents with LLMs**](https://www.nvidia.com/en-sg/training/instructor-led-workshops/building-rag-agents-with-llms/). If sharing this material, please give credit and link back to the original course.

<br>


### **Environment Setup:**

In [2]:
## Necessary for Colab, not necessary for course environment
# %pip install -q langchain langchain-nvidia-ai-endpoints gradio

# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

In [2]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
ChatNVIDIA.get_available_models()

[Model(id='google/gemma-3-4b-it', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=None, supports_tools=False, supports_structured_output=False, supports_thinking=False, base_model=None),
 Model(id='meta/llama3-8b-instruct', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=['ai-llama3-8b'], supports_tools=False, supports_structured_output=False, supports_thinking=False, base_model=None),
 Model(id='mistralai/mixtral-8x22b-instruct-v0.1', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=['ai-mixtral-8x22b-instruct'], supports_tools=False, supports_structured_output=False, supports_thinking=False, base_model=None),
 Model(id='google/gemma-3-12b-it', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=None, supports_tools=False, supports_structured_output=False, supports_thinking=False, base_model=None),
 Model(id='databricks/dbrx-instruct', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=['ai-dbrx-instruct'], supports_to

In [1]:
## Useful utility method for printing intermediate states
from langchain_core.runnables import RunnableLambda
from functools import partial

def RPrint(preface="State: "):
    def print_and_return(x, preface=""):
        print(f"{preface}{x}")
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

def PPrint(preface="State: "):
    def print_and_return(x, preface=""):
        pprint(preface, x)
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

----

<br>

## **Part 1:** Keeping Variables Flowing

In the previous examples, we were able to implement interesting logic in our standalone chains by **creating**, **mutating**, and **consuming** states. These states were passed around as dictionaries with descriptive keys and useful values, and the values would be used to supply follow-up routines with the info they need to operate!

**Recall the zero-shot classification example from the last notebook:**

In [3]:
%%time
## ^^ This notebook is timed, which will print out how long it all took

from langchain_core.runnables import RunnableLambda
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Union
from operator import itemgetter

## Zero-shot classification prompt and chain w/ explicit few-shot prompting
sys_msg = (
    "Choose the most likely topic classification given the sentence as context."
    " Only one word, no explanation.\n[Options : {options}]"
)

# zsc_prompt is a ChatPromptTemplate that formats the prompt for zero-shot classification.
# It uses sys_msg as the system message, which instructs the LLM to choose the most likely topic from the provided options.
# The prompt includes a few-shot example: "[[The sea is awesome]][/INST]boat</s><s>[INST]", which shows the model how to respond with just the topic.
# Then it inserts the actual user input as [[{input}]] for the model to classify.

zsc_prompt = ChatPromptTemplate.from_template(
    f"{sys_msg}\n\n"
    "[[The sea is awesome]][/INST]boat</s><s>[INST]" # ----> prompt format
    "[[{input}]]"
)

# Explanation of special tokens:
# [/INST] and </s> are special tokens used in some LLM prompt formats (like Llama and Mixtral) to indicate the end of an instruction or message.
# - [/INST] marks the end of an instruction block.
# - </s> is a "stop" or "end of sequence" token, often used to separate messages or indicate the end of a response.
# - <s> is a "start of sequence" token, and [INST] marks the start of an instruction.
# In this context, the prompt is formatted to mimic the conversational or instruction-following style expected by the model, helping it understand where the instruction ends and where its answer should begin.

## Define your simple instruct_model
instruct_chat = ChatNVIDIA(model="meta/llama-3.1-405b-instruct")
instruct_llm = instruct_chat | StrOutputParser()
one_word_llm = instruct_chat.bind(stop=[" ", "\n"]) | StrOutputParser()

zsc_chain = zsc_prompt | one_word_llm

## Function that just prints out the first word of the output. With early stopping bind
def zsc_call(input, options=["car", "boat", "airplane", "bike"]):
    result = zsc_chain.invoke({"input" : input, "options" : options})
    words = result.split()
    return words[0] if words else ""

print("-" * 80)
print(zsc_call("Should I take the next exit, or keep going to the next one?"))

print("-" * 80)
print(zsc_call("I get seasick, so I think I'll pass on the trip on water"))

print("-" * 80)
print(zsc_call("I'm scared of heights, so flying on high altitude probably isn't for me"))

--------------------------------------------------------------------------------
car
--------------------------------------------------------------------------------
boat
--------------------------------------------------------------------------------
airplane
CPU times: total: 2.47 s
Wall time: 5.39 s


<br>

This chain makes several design decisions that make it very easy to use, key among them the following:

**We want it to act like a function, so all we want it to do is generate the output and return it.**

This makes the chain extremely natural for inclusion as a module in a larger chain system. For example, the following chain will take a string, extract the most likely topic, and then generate a new sentence based on the topic:



In [8]:
%%time
## ^^ This notebook is timed, which will print out how long it all took
gen_prompt = ChatPromptTemplate.from_template(
    "Make a new sentence about the the following topic: {topic}. Be creative!"
)

gen_chain = gen_prompt | instruct_llm

input_msg = "I get seasick, so I think I'll pass on the trip"
options = ["car", "boat", "airplane", "bike"]

chain = (
    ## -> {"input", "options"}
    {'topic' : zsc_chain}
    | PPrint()
    ## -> {**, "topic"}
    | gen_chain
    ## -> string
)

chain.invoke({"input" : input_msg, "options" : options})

State: 
{'topic': 'boat'}

CPU times: total: 641 ms
Wall time: 2.35 s


'As the sun dipped below the horizon, the old wooden boat creaked to life, its ropes and anchors transformed into a makeshift harp that sang a haunting melody across the waves.'

<br>

However, it's a bit problematic when you want to keep the information flowing, since we lose the topic and input variables in generating our response. If we wanted to do something with both the output and the input, we'd need a way to make sure that both variables pass through.

Lucky for us, we can use the mapping runnable (i.e. interpretted from a dictionary or using manual `RunnableMap`) to pass both of the variables through by assigning the output of our chain to just a single key and letting the other keys propagate as desired. Alternatively, we could also use `RunnableAssign` to merge the state-consuming chain's output with the input dictionary by default.

With this technique, we can propagate whatever we want through our chain system:

In [9]:
%%time
## ^^ This notebook is timed, which will print out how long it all took

from langchain.schema.runnable import RunnableBranch, RunnablePassthrough
from langchain.schema.runnable.passthrough import RunnableAssign
from functools import partial

big_chain = (
    PPrint()
    ## Manual mapping. Can be useful sometimes and inside branch chains
    | {'input' : lambda d: d.get('input'), 'topic' : zsc_chain}
    | PPrint()
    ## RunnableAssign passing. Better for running state chains by default
    | RunnableAssign({'generation' : gen_chain})
    | PPrint()
    ## Using the input and generation together
    | RunnableAssign({'combination' : (
        ChatPromptTemplate.from_template(
            "Consider the following passages:"
            "\nP1: {input}"
            "\nP2: {generation}"
            "\n\nCombine the ideas from both sentences into one simple one."
        )
        | instruct_llm
    )})
)

output = big_chain.invoke({
    "input" : "I get seasick, so I think I'll pass on the trip",
    "options" : ["car", "boat", "airplane", "bike", "unknown"]
})
pprint("Final Output: ", output)

State: 
{
    'input': "I get seasick, so I think I'll pass on the trip",
    'options': ['car', 'boat', 'airplane', 'bike', 'unknown']
}

State: 
{'input': "I get seasick, so I think I'll pass on the trip", 'topic': 'boat'}

State: 
{
    'input': "I get seasick, so I think I'll pass on the trip",
    'topic': 'boat',
    'generation': 'As the sun dipped below the horizon, the old wooden boat creaked to life, its ropes and anchors 
transformed into a makeshift harp that sang a haunting melody across the waves.'
}

Final Output: 
{
    'input': "I get seasick, so I think I'll pass on the trip",
    'topic': 'boat',
    'generation': 'As the sun dipped below the horizon, the old wooden boat creaked to life, its ropes and anchors 
transformed into a makeshift harp that sang a haunting melody across the waves.',
    'combination': "Unfortunately, I'll have to pass on the romantic sailing trip because I get seasick.\n\n(Note: 
I combined the main idea of P1 with the implied setting of P2, which is a sailing trip. I left out the poetic 
details of P2 to keep the new sentence simple.)"
}

CPU times: total: 953 ms
Wall time: 5.19 s


### 🔹 Imports

```python
from langchain.schema.runnable import RunnableBranch, RunnablePassthrough
from langchain.schema.runnable.passthrough import RunnableAssign
from functools import partial
```

* **RunnableBranch** → lets you create *if/else* style chains (not used here but useful later).
* **RunnablePassthrough** → passes data through unchanged (not used here, but often paired with `Assign`).
* **RunnableAssign** → adds new keys to the running state instead of replacing everything.
* **partial** → Python helper for binding arguments into functions.

---

### 🔹 Chain definition

```python
big_chain = (
    PPrint()
    ## Manual mapping. Can be useful sometimes and inside branch chains
    | {'input' : lambda d: d.get('input'), 'topic' : zsc_chain}
    | PPrint()
    ## RunnableAssign passing. Better for running state chains by default
    | RunnableAssign({'generation' : gen_chain})
    | PPrint()
    ## Using the input and generation together
    | RunnableAssign({'combination' : (
        ChatPromptTemplate.from_template(
            "Consider the following passages:"
            "\nP1: {input}"
            "\nP2: {generation}"
            "\n\nCombine the ideas from both sentences into one simple one."
        )
        | instruct_llm
    )})
)
```

Let’s break it down:

---

#### Step 1: `PPrint()`

* Prints the incoming state for debugging.
* Initial input is:

  ```python
  {
    "input": "I get seasick, so I think I'll pass on the trip",
    "options": ["car", "boat", "airplane", "bike", "unknown"]
  }
  ```

---

#### Step 2: Manual mapping

```python
| {'input' : lambda d: d.get('input'), 'topic' : zsc_chain}
```

* This takes the state and builds a **new dictionary**:

  * `"input"` is just copied from before.
  * `"topic"` is generated by `zsc_chain` (a classifier).
* Example result:

  ```python
  {
    "input": "I get seasick, so I think I'll pass on the trip",
    "topic": "boat"
  }
  ```

---

#### Step 3: Assign generation

```python
| RunnableAssign({'generation' : gen_chain})
```

* Instead of replacing the whole state, this **adds** a new key:
  `"generation"` = output of `gen_chain`, a sentence generator.
* Example result:

  ```python
  {
    "input": "I get seasick, so I think I'll pass on the trip",
    "topic": "boat",
    "generation": "A sleek boat glided silently across the waves."
  }
  ```

---

#### Step 4: Assign combination

```python
| RunnableAssign({'combination' : (
    ChatPromptTemplate.from_template(
        "Consider the following passages:"
        "\nP1: {input}"
        "\nP2: {generation}"
        "\n\nCombine the ideas from both sentences into one simple one."
    )
    | instruct_llm
)})
```

* Another assignment step.
* This time, it uses both `{input}` and `{generation}` in a prompt.
* The LLM fuses them into a single simple sentence.
* Example result:

  ```python
  {
    "input": "I get seasick, so I think I'll pass on the trip",
    "topic": "boat",
    "generation": "A sleek boat glided silently across the waves.",
    "combination": "Although I get seasick, I admire how gracefully boats move across the water."
  }
  ```

---

### 🔹 Invocation

```python
output = big_chain.invoke({...})
pprint("Final Output: ", output)
```

* Runs the whole chain with your input.
* Prints each intermediate state (`PPrint()`), then prints the final output dictionary.

---

### 🔹 Summary

This chain does 3 things:

1. **Classify** input into a topic (boat).
2. **Generate** a creative sentence about that topic.
3. **Combine** the original input + generated text into a new sentence.

---

👉 The cool thing:

* If you used `|` directly, each step would **replace** the state.
* But with `RunnableAssign`, you **grow** the state by adding new fields.
* That’s why at the end you have all of `input`, `topic`, `generation`, and `combination` together.



----

<br>

## **Part 2:** Running State Chain

The example above is just a toy example and, if anything, showcases the drawbacks of chaining many LLM calls together for internal under-the-hood reasoning. However, the ability to keep information flowing through a chain is invaluable for making complex chains that can accumulate useful state information or operate in a multi-pass capacity.

Specifically, a very simple but effective chain is a **Running State Chain** which enforces the following properties:
- A **"running state"** is a dictionary that contains all of the variables that the system cares about.
- A **"branch"** is a chain that can pull in the running state and can degenerate it into a response.
- A **branch** can only be ran inside a **RunnableAssign** scope, and the branchs' inputs should come from the **running state**.

> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/running_state_chain.png" width=1000px/>
<!-- > <img src="https://drive.google.com/uc?export=view&id=1Oo7AauYGj4dxepNReRG2JezmvQLyqXsN" width=1000px/> -->

You can think of the running state chain abstraction as a functional variant of a Pythonic class with state variables (or attributes) and functions (or methods).
- The chain is like the abstract class that wraps all of the functionality.
- The running state are like the attributes (which should always be accessible).
- The branches are like the class methods (which can pick and choose which attributes to use).
- The `.invoke` or similar process is like the `__call__` method that runs through the branches in order.

**By forcing this paradigm in your chains:**
- You can keep state variables propagating through your chain, allowing your internals to access whatever is necessary and accumulating state values for use later.
- You can also pass the outputs of your chain back through as your inputs, allowing a "while-loop"-style chain that keeps updating and building on your running state.

The rest of this notebook will include two exercises that flesh out the running state chain abstraction for two additional use-cases: **Knowledge Bases** and **Database-Querying Chatbots**.

----

<br>

## **Part 3:** Implementing a Knowledge Base with Running State Chain

After understanding the basic structure and principles of a Running State Chain, we can explore how this approach can be extended to manage more complex tasks, particularly in creating dynamic systems that evolve through interaction. This section will focus on implementing a **knowledge base** accumulated using **json-enabled slot filling**:

- **Knowledge Base:** A store of information that's relevant for our LLM to keep track of.
- **JSON-Enabled Slot Filling:** The technique of asking an instruction-tuned model to output a json-style format (which can include a dictionary) with a selection of slots, relying on the LLM to fill these slots with useful and relevant information.

<br>

#### **Defining Our Knowledge Base**

To build a responsive and intelligent system, we need a method that not only processes inputs but also retains and updates essential information through the flow of conversation. This is where the combination of LangChain and Pydantic becomes pivotal. [**Pydantic**](https://docs.pydantic.dev/latest/), a popular Python validation library, is instrumental in structuring and validating data models. As one of its features, Pydantic offers structured "model" classes that validate objects (data, classes, themselves, etc.) with simplified syntax and deep rabbitholes of customization options. This framework is used throughout LangChain and comes up as a necessary component for use cases that involve data coersion.

One thing that a "model" is very good for is defining a class with expected arguments and some special ways to validate them! In this course, we won't focus too much on the validation scripts, but those interested can start by checking out the [**Pydantic Validator guide**](https://docs.pydantic.dev/1.10/usage/validators/) (though the topics do get pretty deep pretty fast). For our purposes, we can construct a `BaseModel` class and define some `Field` variables to create a structured **Knowledge Base** like so:

In [10]:
from pydantic import BaseModel, Field
from typing import Dict, Union, Optional

class KnowledgeBase(BaseModel):
    ## Fields of the BaseModel, which will be validated/assigned when the knowledge base is constructed
    topic: str = Field('general', description="Current conversation topic")
    user_preferences: Dict[str, Union[str, int]] = Field({}, description="User preferences and choices")
    session_notes: str = Field("", description="Notes on the ongoing session")
    unresolved_queries: list = Field([], description="Unresolved user queries")
    action_items: list = Field([], description="Actionable items identified during the conversation")

print(repr(KnowledgeBase(topic = "Machine learning")))

KnowledgeBase(topic='Machine learning', user_preferences={}, session_notes='', unresolved_queries=[], action_items=[])


<br>

The true strength of this approach lies in the additional LLM-centric functionalities provided by LangChain which we can integrate for our use-cases. One such feature is the `PydanticOutputParser` which enhances the Pydantic objects with capabilities like automatic format instruction generation.

In [11]:
from langchain.output_parsers import PydanticOutputParser

instruct_string = PydanticOutputParser(pydantic_object=KnowledgeBase).get_format_instructions()
pprint(instruct_string)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": 
"array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": 
["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"topic": {"default": "general", "description": "Current conversation topic", "title": "Topic", 
"type": "string"}, "user_preferences": {"additionalProperties": {"anyOf": [{"type": "string"}, {"type": 
"integer"}]}, "default": {}, "description": "User preferences and choices", "title": "User Preferences", "type": 
"object"}, "session_notes": {"default": "", "description": "Notes on the ongoing session", "title": "Session 
Notes", "type": "string"}, "unresolved_queries": {"default": [], "description": "Unresolved user queries", "items":
{}, "title": "Unresolved Queries", "type": "array"}, "action_items": {"default": [], "description": "Actionable 
items identified during the conversation", "items": {}, "title": "Action Items", "type": "array"}}}
```

In [12]:
from pprint import pprint as pretty_print

t={"properties": {"topic": {"default": "general", "description": "Current conversation topic", "title": "Topic", 
"type": "string"}, "user_preferences": {"additionalProperties": {"anyOf": [{"type": "string"}, {"type": 
"integer"}]}, "default": {}, "description": "User preferences and choices", "title": "User Preferences", "type": 
"object"}, "session_notes": {"default": "", "description": "Notes on the ongoing session", "title": "Session Notes", "type": "string"}, "unresolved_queries": {"default": [], "description": "Unresolved user queries", "items":
{}, "title": "Unresolved Queries", "type": "array"}, "action_items": {"default": [], "description": "Actionable items identified during the conversation", "items": {}, "title": "Action Items", "type": "array"}}}

pretty_print(t)

{'properties': {'action_items': {'default': [],
                                 'description': 'Actionable items identified '
                                                'during the conversation',
                                 'items': {},
                                 'title': 'Action Items',
                                 'type': 'array'},
                'session_notes': {'default': '',
                                  'description': 'Notes on the ongoing session',
                                  'title': 'Session Notes',
                                  'type': 'string'},
                'topic': {'default': 'general',
                          'description': 'Current conversation topic',
                          'title': 'Topic',
                          'type': 'string'},
                'unresolved_queries': {'default': [],
                                       'description': 'Unresolved user queries',
                                       'items': {},
     

This functionality generates instructions for creating valid inputs to the Knowledge Base, which in turn helps the LLM by providing a concrete one-shot example of the desired output format.

<br>

#### **Runnable Extraction Module**

Knowing that we have this Pydantic object which can be used to generate good LLM instructions, we can make a Runnable that wraps the functionality of our Pydantic class and streamlines the prompting, generating, and updating of the knowledge base:

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema.runnable.passthrough import RunnableAssign
################################################################################
## Definition of RExtract
def RExtract(pydantic_class, llm, prompt):
    '''
    Runnable Extraction module
    Returns a knowledge dictionary populated by slot-filling extraction
    '''
    parser = PydanticOutputParser(pydantic_object=pydantic_class)
    instruct_merge = RunnableAssign({'format_instructions' : lambda x: parser.get_format_instructions()})
    def preparse(string):
        if '{' not in string: string = '{' + string
        if '}' not in string: string = string + '}'
        string = (string
            .replace("\\_", "_")
            .replace("\n", " ")
            .replace("\]", "]")
            .replace("\[", "[")
        )
        # print(string)  ## Good for diagnostics
        return string
    return instruct_merge | prompt | llm | preparse | parser

################################################################################
## Practical Use of RExtract

parser_prompt = ChatPromptTemplate.from_template(
    "Update the knowledge base: {format_instructions}. Only use information from the input."
    "\n\nNEW MESSAGE: {input}"
)

extractor = RExtract(KnowledgeBase, instruct_llm, parser_prompt)

knowledge = extractor.invoke({'input' : "I love flowers so much! The orchids are amazing! Can you buy me some?"})
pprint(knowledge)

<>:18: SyntaxWarning: invalid escape sequence '\]'
<>:19: SyntaxWarning: invalid escape sequence '\['
<>:18: SyntaxWarning: invalid escape sequence '\]'
<>:19: SyntaxWarning: invalid escape sequence '\['
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_75796\3420359838.py:18: SyntaxWarning: invalid escape sequence '\]'
  .replace("\]", "]")
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_75796\3420359838.py:19: SyntaxWarning: invalid escape sequence '\['
  .replace("\[", "[")
<unknown>:7: SyntaxWarning: invalid escape sequence '\]'
<unknown>:8: SyntaxWarning: invalid escape sequence '\['


KnowledgeBase(
    topic='general',
    user_preferences={'favorite_flowers': 'orchids'},
    session_notes='',
    unresolved_queries=['buy orchids'],
    action_items=[]
)

<br>

Do keep in mind that this process can fail due to the fuzzy nature of LLM prediction, especially with models that are not optimized for instruction-following! For this process, it's important to have a strong instruction-following LLM with extra checks and graceful failure routines. 

<br>

#### **Dynamic Knowledge Base Updates**

Finally, we can create a system that continually updates the Knowledge Base throughout the conversation. This is done by feeding the current state of the Knowledge Base, along with new user inputs, back into the system for ongoing updates.

The following is an example system that shows off both the formulation's power of filling details as well as the limitations of assuming that filling performance will be as good as general response performance:

In [15]:
from langchain_core.output_parsers import StrOutputParser

class KnowledgeBase(BaseModel):
    firstname: str = Field('unknown', description="Chatting user's first name, unknown if unknown")
    lastname: str = Field('unknown', description="Chatting user's last name, unknown if unknown")
    location: str = Field('unknown', description="Where the user is located")
    summary: str = Field('unknown', description="Running summary of conversation. Update this with new input")
    response: str = Field('unknown', description="An ideal response to the user based on their new message")


parser_prompt = ChatPromptTemplate.from_template(
    "You are chatting with a user. The user just responded ('input'). Please update the knowledge base."
    " Record your response in the 'response' tag to continue the conversation."
    " Do not hallucinate any details, and make sure the knowledge base is not redundant."
    " Update the entries frequently to adapt to the conversation flow."
    "\n{format_instructions}"
    "\n\nOLD KNOWLEDGE BASE: {know_base}"
    "\n\nNEW MESSAGE: {input}"
    "\n\nNEW KNOWLEDGE BASE:"
)

## Switch to a more powerful base model
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1") | StrOutputParser()

extractor = RExtract(KnowledgeBase, instruct_llm, parser_prompt)
info_update = RunnableAssign({'know_base' : extractor})

## Initialize the knowledge base and see what you get
state = {'know_base' : KnowledgeBase()}
state['input'] = "My name is Carmen Sandiego! Guess where I am! Hint: It's somewhere in the United States."
state = info_update.invoke(state)
pprint(state)

<unknown>:7: SyntaxWarning: invalid escape sequence '\]'
<unknown>:8: SyntaxWarning: invalid escape sequence '\['


{
    'know_base': KnowledgeBase(
        firstname='Carmen',
        lastname='Sandiego',
        location='unknown',
        summary='The user introduced themselves as Carmen Sandiego and asked for a guess on their location within 
the United States, providing a hint.',
        response="Welcome, Carmen Sandiego! I'm excited to try and guess your location. Since you mentioned it's 
somewhere in the United States, I'll start there. Is it by any chance in New York?"
    ),
    'input': "My name is Carmen Sandiego! Guess where I am! Hint: It's somewhere in the United States."
}

In [ ]:
state['input'] = "I'm in a place considered the birthplace of Jazz."
state = info_update.invoke(state)
pprint(state)

{
    'know_base': KnowledgeBase(
        firstname='Carmen',
        lastname='Sandiego',
        location='unknown',
        summary='The user introduced themselves as Carmen Sandiego and asked for a guess on their location within 
the United States, providing a hint. The user has now indicated that they are in a place considered the birthplace 
of Jazz.',
        response='Interesting hint, Carmen Sandiego! I believe you might be in New Orleans, a city in the United 
States known as the birthplace of Jazz. Is that correct?'
    ),
    'input': "I'm in a place considered the birthplace of Jazz."
}

In [22]:
state['input'] = "Yeah, I'm in New Orleans... How did you know?"
state = info_update.invoke(state)
pprint(state)

{
    'know_base': KnowledgeBase(
        firstname='Carmen',
        lastname='Sandiego',
        location='New Orleans',
        summary='The user introduced themselves as Carmen Sandiego and asked for a guess on their location within 
the United States, providing a hint. The user has now confirmed that they are in New Orleans, a city in the United 
States known as the birthplace of Jazz.',
        response='I guessed correctly based on your hint, Carmen Sandiego. New Orleans is indeed known as the 
birthplace of Jazz.'
    ),
    'input': "Yeah, I'm in New Orleans... How did you know?"
}

<br>

This example demonstrates how a running state chain can be effectively utilized to manage a conversation with evolving context and requirements, making it a powerful tool for developing sophisticated interactive systems.

The next sections of this notebook will expand on these concepts by exploring two specific applications: **Document Knowledge Bases** and **Database-Querying Chatbots**.

----

<br>

## **Part 4: [Exercise]** Airline Customer Service Bot

In this exercise, we can expand on the tools we've learned about to implement a simple but effective dialog manager chatbot. For this exercise, we will make an airline support bot that wants to help a client find out about their flight!

Let's create a simple database-like interface to get some customer information from a dictionary!

In [28]:
from pydantic import BaseModel, Field
from typing import Optional

# knowledge base
class AirlineInfo(BaseModel):
#Field is being used for clear instruction for LLM
    firstname:str = Field("unknown", description="Customer's first name",min_length=1,max_length=50,examples=["Rahul","Anjali"])
    lastname: str = Field("unknown", description="Customer's last name",min_length=1,max_length=50,examples=["Khanna","Sharma"])
    flight_number: str = Field("unknown", description="Customer's flight number")
    departure_city: str = Field("unknown", description="City the passenger is departing from")
    arrival_city: str = Field("unknown", description="City the passenger is arriving at")
    departure_time: str = Field("unknown", description="Scheduled departure time")
    arrival_time: str = Field("unknown", description="Scheduled arrival time")
    summary: str = Field("unknown", description="Running summary of the conversation so far")
    intent: str = Field("unknown", description="What the customer wants (status, time, destination, etc.)",examples="status")
    response: str = Field("unknown", description="Bot's response to the customer",max_length=714)

#create a database
airline_db = {
    "AA123": {"destination": "New York", "departure": "Los Angeles", "time": "10:00 AM"},
    "BA456": {"destination": "London", "departure": "New York", "time": "3:00 PM"},
    "DL789": {"destination": "Paris", "departure": "Atlanta", "time": "7:00 PM"},
}


In [29]:
# prompt for slot filling

from langchain_core.prompts import ChatPromptTemplate

external_prompt = ChatPromptTemplate.from_template( 
    "You are SkyFlow’s virtual airline assistant. Your role is to help the customer with their request."
    " Respond in a professional, concise, and friendly way — short and sweet when possible."
    " Do NOT greet unless necessary (e.g., only if the customer greets first)."
    " SkyFlow uses industry-average airline practices for operations and arrival times."
    " Never disclose this fact or mention the phrase 'industry average practices' to the user."
    "\n\nRelevant Context: {context}"
    "\n\nCustomer: {input}"
    "\n\nAssistant Response:"
    " Always return a JSON that follows the format instructions."
    "\n{format_instructions}"
    "\n\nOLD KNOWLEDGE BASE: {know_base}"
    "\n\nNEW MESSAGE: {input}"
    "\n\nNEW KNOWLEDGE BASE:"
)

In [30]:
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable.passthrough import RunnableAssign

# Stronger LLM
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1") | StrOutputParser()

# Extractor with airline schema
extractor = RExtract(AirlineInfo, instruct_llm, external_prompt)

# Running state manager
dialog_manager = RunnableAssign({'know_base': extractor})


In [31]:
from langchain.output_parsers import PydanticOutputParser
from langchain.schema.runnable.passthrough import RunnableAssign
from langchain.schema.runnable import RunnableLambda
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser

# LLM
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1") | StrOutputParser()

def RExtract(pydantic_class, llm, prompt):
    parser = PydanticOutputParser(pydantic_object=pydantic_class)
    instruct_merge = RunnableAssign({'format_instructions': lambda x: parser.get_format_instructions()})
    def preparse(string):
        if '{' not in string: string = '{' + string
        if '}' not in string: string = string + '}'
        return string.replace("\n", " ")
    return instruct_merge | prompt | llm | preparse | parser


In [32]:
def lookup_flight(kb: AirlineInfo) -> Dict:
    if kb.flight_number in airline_db:
        details = airline_db[kb.flight_number]
        return {"response": f"Your flight {kb.flight_number} departs from {details['departure']} "
                            f"to {details['destination']} at {details['time']}."}
    else:
        return {"response": "Could you please confirm your flight number?"}


In [33]:
# Create extractor
extractor = RExtract(AirlineInfo, instruct_llm, parser_prompt)

# Add to running state
dialog_manager = (
    RunnableAssign({"know_base": extractor})
    | RunnableAssign({"know_base": lambda state: state["know_base"].copy(update=lookup_flight(state["know_base"]))})
)


In [34]:
# Initial state
state = {"know_base": AirlineInfo()}

# User 1
state["input"] = "Hi, my name is Alice Johnson."
state = dialog_manager.invoke(state)
print(state["know_base"])

# User 2
state["input"] = "My flight number is AA123."
state = dialog_manager.invoke(state)
print(state["know_base"])

# User 3
state["input"] = "Can you tell me when it leaves?"
state = dialog_manager.invoke(state)
print(state["know_base"])


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_75796\3512348316.py:7: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  | RunnableAssign({"know_base": lambda state: state["know_base"].copy(update=lookup_flight(state["know_base"]))})


firstname='Alice' lastname='Johnson' flight_number='unknown' departure_city='unknown' arrival_city='unknown' departure_time='unknown' arrival_time='unknown' summary='The customer has introduced herself as Alice Johnson.' intent='unknown' response='Could you please confirm your flight number?'


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_75796\3512348316.py:7: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  | RunnableAssign({"know_base": lambda state: state["know_base"].copy(update=lookup_flight(state["know_base"]))})


firstname='Alice' lastname='Johnson' flight_number='AA123' departure_city='unknown' arrival_city='unknown' departure_time='unknown' arrival_time='unknown' summary='The customer has introduced herself as Alice Johnson. She has provided her flight number as AA123.' intent='unknown' response='Your flight AA123 departs from Los Angeles to New York at 10:00 AM.'
firstname='Alice' lastname='Johnson' flight_number='AA123' departure_city='unknown' arrival_city='unknown' departure_time='unknown' arrival_time='unknown' summary='The customer has introduced herself as Alice Johnson. She has provided her flight number as AA123. The customer wants to know the departure time.' intent='departure time' response='Your flight AA123 departs from Los Angeles to New York at 10:00 AM.'


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_75796\3512348316.py:7: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  | RunnableAssign({"know_base": lambda state: state["know_base"].copy(update=lookup_flight(state["know_base"]))})


---

In [9]:
#######################################################################################
## Function that can be queried for information. Implementation details not important
def get_flight_info(d: dict) -> str:
    """
    Example of a retrieval function which takes a dictionary as key. Resembles SQL DB Query
    """
    req_keys = ['first_name', 'last_name', 'confirmation']
    assert all((key in d) for key in req_keys), f"Expected dictionary with keys {req_keys}, got {d}"

    ## Static dataset. get_key and get_val can be used to work with it, and db is your variable
    keys = req_keys + ["departure", "destination", "departure_time", "arrival_time", "flight_day"]
    values = [
        ["Jane", "Doe", 12345, "San Jose", "New Orleans", "12:30 PM", "9:30 PM", "tomorrow"],
        ["John", "Smith", 54321, "New York", "Los Angeles", "8:00 AM", "11:00 AM", "Sunday"],
        ["Alice", "Johnson", 98765, "Chicago", "Miami", "7:00 PM", "11:00 PM", "next week"],
        ["Bob", "Brown", 56789, "Dallas", "Seattle", "1:00 PM", "4:00 PM", "yesterday"],
    ]
    get_key = lambda d: "|".join([d['first_name'], d['last_name'], str(d['confirmation'])])
    get_val = lambda l: {k:v for k,v in zip(keys, l)}
    db = {get_key(get_val(entry)) : get_val(entry) for entry in values}

    # Search for the matching entry
    data = db.get(get_key(d))
    if not data:
        return (
            f"Based on {req_keys} = {get_key(d)}) from your knowledge base, no info on the user flight was found."
            " This process happens every time new info is learned. If it's important, ask them to confirm this info."
        )
    return (
        f"{data['first_name']} {data['last_name']}'s flight from {data['departure']} to {data['destination']}"
        f" departs at {data['departure_time']} {data['flight_day']} and lands at {data['arrival_time']}."
    )

#######################################################################################
## Usage example. Actually important

print(get_flight_info({"first_name" : "Jane", "last_name" : "Doe", "confirmation" : 12345}))

Jane Doe's flight from San Jose to New Orleans departs at 12:30 PM tomorrow and lands at 9:30 PM.


In [10]:
print(get_flight_info({"first_name" : "Alice", "last_name" : "Johnson", "confirmation" : 98765}))

Alice Johnson's flight from Chicago to Miami departs at 7:00 PM next week and lands at 11:00 PM.


In [11]:
print(get_flight_info({"first_name" : "Bob", "last_name" : "Brown", "confirmation" : 27494}))

Based on ['first_name', 'last_name', 'confirmation'] = Bob|Brown|27494) from your knowledge base, no info on the user flight was found. This process happens every time new info is learned. If it's important, ask them to confirm this info.


<br>

This is a really good interface to bring up because it can reasonably serve two purposes:
- It can be used to provide up-to-date information from an external environment (a database) regarding a user's situation.
- It can also be used as a hard gating mechanism to prevent unauthorized disclosure of sensitive information (since that would be very bad).

If our network had access to this kind of interface, it would be able to query for and retrieve this information on a user's behalf! For example:

In [12]:
external_prompt = ChatPromptTemplate.from_template(
    "You are a SkyFlow chatbot, and you are helping a customer with their issue."
    " Please help them with their question, remembering that your job is to represent SkyFlow airlines."
    " Assume SkyFlow uses industry-average practices regarding arrival times, operations, etc."
    " (This is a trade secret. Do not disclose)."  ## soft reinforcement
    " Please keep your discussion short and sweet if possible. Avoid saying hello unless necessary."
    " The following is some context that may be useful in answering the question."
    "\n\nContext: {context}"
    "\n\nUser: {input}"
)

basic_chain = external_prompt | instruct_llm

basic_chain.invoke({
    'input' : 'Can you please tell me when I need to get to the airport?',
    'context' : get_flight_info({"first_name" : "Jane", "last_name" : "Doe", "confirmation" : 12345}),
})

'Jane, your flight departs at 12:30 PM. For domestic flights, we recommend arriving at least 2 hours prior to your scheduled departure time. In this case, you should plan to arrive at San Jose airport by 10:30 AM.'

<br>

This is interesting enough, but how do we actually get this system working in the wild? It turns out that we can use the KnowledgeBase formulation from above to supply this kind of information like so:

In [13]:
from langchain.pydantic_v1 import BaseModel, Field
from typing import Dict, Union

class KnowledgeBase(BaseModel):
    first_name: str = Field('unknown', description="Chatting user's first name, `unknown` if unknown")
    last_name: str = Field('unknown', description="Chatting user's last name, `unknown` if unknown")
    confirmation: int = Field(-1, description="Flight Confirmation Number, `-1` if unknown")
    discussion_summary: str = Field("", description="Summary of discussion so far, including locations, issues, etc.")
    open_problems: list = Field([], description="Topics that have not been resolved yet")
    current_goals: list = Field([], description="Current goal for the agent to address")

def get_key_fn(base: BaseModel) -> dict:
    '''Given a dictionary with a knowledge base, return a key for get_flight_info'''
    return {  ## More automatic options possible, but this is more explicit
        'first_name' : base.first_name,
        'last_name' : base.last_name,
        'confirmation' : base.confirmation,
    }

know_base = KnowledgeBase(first_name = "Jane", last_name = "Doe", confirmation = 12345)

# get_flight_info(get_key_fn(know_base))

get_key = RunnableLambda(get_key_fn)
(get_key | get_flight_info).invoke(know_base)

"Jane Doe's flight from San Jose to New Orleans departs at 12:30 PM tomorrow and lands at 9:30 PM."

<br>

### **Objective:**

You want a user to be able to invoke the following function call organically as part of a dialog exchange:

```python
get_flight_info({"first_name" : "Jane", "last_name" : "Doe", "confirmation" : 12345}) ->
    "Jane Doe's flight from San Jose to New Orleans departs at 12:30 PM tomorrow and lands at 9:30 PM."
```

`RExtract` is provided such that the following knowledge base syntax can be used:
```python
known_info = KnowledgeBase()
extractor = RExtract(KnowledgeBase, InstructLLM(), parser_prompt)
results = extractor.invoke({'info_base' : known_info, 'input' : 'My message'})
known_info = results['info_base']
```

**Design a chatbot that implements the following features:**
- The bot should start off by making small-talk, possibly helping the user with non-sensitive queries which don't require any private info access.
- When the user starts to ask about things that are database-walled (both practically and legally), tell the user that they need to provide the relevant information.
- When the retrieval is successful, the agent will be able to talk about the database-walled information.

**This can be done with a variety of techniques, including the following:**
- **Prompt Engineering and Context Parsing**, where the overall chat prompt stays roughly the same but the context is manipulated to to change agent behavior. For example, a failed db retrieval could be changed into an injection of natural-language instructions for how to resolve the problem such as *`"Information could not be retrieved with keys {...}. Please ask the user for clarification or help them with known information."`*
- **"Prompt Passing,"** where the active prompts are passed around as state variables and can be overridden by monitoring chains.
- **Branching chains** such as [**`RunnableBranch`**](https://api.python.langchain.com/en/latest/core/runnables/langchain_core.runnables.branch.RunnableBranch.html) or more custom solutions that implement an conditional routing mechanism.
    - In the case of [`RunnableBranch`](https://api.python.langchain.com/en/latest/core/runnables/langchain_core.runnables.branch.RunnableBranch.html), a `switch` syntax of the style:
        ```python
        from langchain.schema.runnable import RunnableBranch
        RunnableBranch(
            ((lambda x: 1 in x), RPrint("Has 1 (didn't check 2): ")),
            ((lambda x: 2 in x), RPrint("Has 2 (not 1 though): ")),
            RPrint("Has neither 1 not 2: ")
        ).invoke([2, 1, 3]);  ## -> Has 1 (didn't check 2): [2, 1, 3]
        ```

Some prompts and a gradio loop are provided that might help with the effort, but the agent will currently just hallucinate! Please implement the internal chain to try and retrieve the relevant information. Before trying to implement, look over the default behavior of the model and note how it might hallucinate or forget things.

In [14]:
from langchain.schema.runnable import (
    RunnableBranch,
    RunnableLambda,
    RunnableMap,       ## Wrap an implicit "dictionary" runnable
    RunnablePassthrough,
)
from langchain.schema.runnable.passthrough import RunnableAssign
from pydantic import BaseModel,Field
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, SystemMessage, ChatMessage, AIMessage
from typing import Iterable
import gradio as gr
from typing import Optional

external_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a chatbot for SkyFlow Airlines, and you are helping a customer with their issue."
        " Please chat with them! Stay concise and clear!"
        " Your running knowledge base is: {know_base}."
        " This is for you only; Do not mention it!"
        " \nUsing that, we retrieved the following: {context}\n"
        " If they provide info and the retrieval fails, ask to confirm their first/last name and confirmation."
        " Do not ask them any other personal info."
        " If it's not important to know about their flight, do not ask."
        " The checking happens automatically; you cannot check manually."
    )),
    ("assistant", "{output}"),
    ("user", "{input}"),
])

##########################################################################
## Knowledge Base Things

class KnowledgeBase(BaseModel):
    first_name: str = Field('unknown', description="Chatting user's first name, `unknown` if unknown")
    last_name: str = Field('unknown', description="Chatting user's last name, `unknown` if unknown")
    confirmation: Optional[int] = Field(None, description="Flight Confirmation Number, `-1` if unknown")
    discussion_summary: str = Field("", description="Summary of discussion so far, including locations, issues, etc.")
    open_problems: str = Field("", description="Topics that have not been resolved yet")
    current_goals: str = Field("", description="Current goal for the agent to address")

parser_prompt = ChatPromptTemplate.from_template(
    "You are a chat assistant representing the airline SkyFlow, and are trying to track info about the conversation."
    " You have just received a message from the user. Please fill in the schema based on the chat."
    "\n\n{format_instructions}"
    "\n\nOLD KNOWLEDGE BASE: {know_base}"
    "\n\nASSISTANT RESPONSE: {output}"
    "\n\nUSER MESSAGE: {input}"
    "\n\nNEW KNOWLEDGE BASE: "
)

## Your goal is to invoke the following through natural conversation
# get_flight_info({"first_name" : "Jane", "last_name" : "Doe", "confirmation" : 12345}) ->
#     "Jane Doe's flight from San Jose to New Orleans departs at 12:30 PM tomorrow and lands at 9:30 PM."

chat_llm = ChatNVIDIA(model="meta/llama-3.1-405b-instruct") | StrOutputParser()
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1") | StrOutputParser()

external_chain = external_prompt | chat_llm

#####################################################################################
## START TODO: Define the extractor and internal chain to satisfy the objective

## TODO: Make a chain that will populate your knowledge base based on provided context
knowbase_getter = lambda x: KnowledgeBase()

## TODO: Make a chain to pull d["know_base"] and outputs a retrieval from db
database_getter = lambda x: "Not implemented"

## These components integrate to make your internal chain
internal_chain = (
    RunnableAssign({'know_base' : knowbase_getter})
    | RunnableAssign({'context' : database_getter})
)

## END TODO
#####################################################################################

state = {'know_base' : KnowledgeBase()}

def chat_gen(message, history=[], return_buffer=True):

    ## Pulling in, updating, and printing the state
    global state
    state['input'] = message
    state['history'] = history
    state['output'] = "" if not history else history[-1][1]

    ## Generating the new state from the internal chain
    state = internal_chain.invoke(state)
    print("State after chain run:")
    pprint({k:v for k,v in state.items() if k != "history"})
    
    ## Streaming the results
    buffer = ""
    for token in external_chain.stream(state):
        buffer += token
        yield buffer if return_buffer else token

def queue_fake_streaming_gradio(chat_stream, history = [], max_questions=8):

    ## Mimic of the gradio initialization routine, where a set of starter messages can be printed off
    for human_msg, agent_msg in history:
        if human_msg: print("\n[ Human ]:", human_msg)
        if agent_msg: print("\n[ Agent ]:", agent_msg)

    ## Mimic of the gradio loop with an initial message from the agent.
    for _ in range(max_questions):
        message = input("\n[ Human ]: ")
        print("\n[ Agent ]: ")
        history_entry = [message, ""]
        for token in chat_stream(message, history, return_buffer=False):
            print(token, end='')
            history_entry[1] += token
        history += [history_entry]
        print("\n")

## history is of format [[User response 0, Bot response 0], ...]
chat_history = [[None, "Hello! I'm your SkyFlow agent! How can I help you?"]]

## Simulating the queueing of a streaming gradio interface, using python input
queue_fake_streaming_gradio(
    chat_stream = chat_gen,
    history = chat_history
)


[ Agent ]: Hello! I'm your SkyFlow agent! How can I help you?

[ Agent ]: 
State after chain run:


{
    'know_base': KnowledgeBase(
        first_name='unknown',
        last_name='unknown',
        confirmation=None,
        discussion_summary='',
        open_problems='',
        current_goals=''
    ),
    'input': 'my name is john what is my flight conformed ',
    'output': "Hello! I'm your SkyFlow agent! How can I help you?",
    'context': 'Not implemented'
}

Hello John! Unfortunately, I couldn't retrieve your flight information. Can you please confirm your last name and also provide me with your confirmation number so I can better assist you?


[ Agent ]: 
State after chain run:


{
    'know_base': KnowledgeBase(
        first_name='unknown',
        last_name='unknown',
        confirmation=None,
        discussion_summary='',
        open_problems='',
        current_goals=''
    ),
    'input': 'jhon dove flight number 12345',
    'output': "Hello John! Unfortunately, I couldn't retrieve your flight information. Can you please confirm your 
last name and also provide me with your confirmation number so I can better assist you?",
    'context': 'Not implemented'
}

I couldn't retrieve your flight information. Can you please confirm if your first name is spelled "Jhon" and your last name is "Dove"? Also, is "12345" your confirmation number?


[ Agent ]: 
State after chain run:


{
    'know_base': KnowledgeBase(
        first_name='unknown',
        last_name='unknown',
        confirmation=None,
        discussion_summary='',
        open_problems='',
        current_goals=''
    ),
    'input': 'yes',
    'output': 'I couldn\'t retrieve your flight information. Can you please confirm if your first name is spelled 
"Jhon" and your last name is "Dove"? Also, is "12345" your confirmation number?',
    'context': 'Not implemented'
}

Thank you for confirming. Unfortunately, I still couldn't retrieve your flight information. Can you please tell me a little bit about the issue you're experiencing so I can better assist you?



KeyboardInterrupt: Interrupted by user

In [ ]:
# state = {'know_base' : KnowledgeBase()}

# chatbot = gr.Chatbot(value=[[None, "Hello! I'm your SkyFlow agent! How can I help you?"]])
# demo = gr.ChatInterface(chat_gen, chatbot=chatbot).queue().launch(debug=True, share=True)

<br>

----

<br>

**NOTE:**
- You may need to explicitly hit the STOP button and try to relaunch your gradio interface if it hangs up after an exception. This is a known Jupyter Notebook environment issue which should not be experienced in dedicated Gradio-running files.
- **Your chat directive is duplicated here for quick access:**
```python
## Your goal is to invoke the following through natural conversation
get_flight_info({
    "first_name" : "Jane",
    "last_name" : "Doe",
    "confirmation" : 12345,
}) -> "Jane Doe's flight from San Jose to New Orleans departs at 12:30 PM tomorrow and lands at 9:30 PM."
```
- **To confirm that your system works, you could try the following dialog or something similar:**
```
> How's it going?
> Can you tell me a bit about skyflow?
> Can you tell me about my flight?
> My name is Jane Doe and my flight confirmation is 12345
> Can you tell me when I should get to my flight?
```
- **Solutions To Exercises Can Be Found In The Solutions Directory.** This is the first exercise with a noted solution, and additional exercises from the future notebooks will be found there.

-----

<br>

## **Part 5:** Wrap-Up

The goal of this notebook was to introduce some more advanced LangChain material revolving around the use of knowledge bases and running state chains! The exercise here was pretty involved, so congrats on finishing it!

### <font color="#76b900">**Great Job!**</font>

### **Next Steps:**
1. **[Optional]** Revisit the **"Questions To Think About" Section** at the top of the notebook and think about some possible answers.

---

<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>

# Practice learning

In [8]:
from langchain.schema.runnable import RunnableLambda

# Step: uppercase a string
to_upper = RunnableLambda(lambda x: x.upper())

# Stream the result (character by character, as state evolves)
for chunk in to_upper.stream("hello"):
    print("Chunk:", chunk)

type(to_upper)

Chunk: HELLO


langchain_core.runnables.base.RunnableLambda

Let’s tackle **`RunnableConfig` and `RunnableState` (often referred to as “running state”)** so you see where it fits in.

---

## 🔹 What is `RunnableState` (or running state)?

When you run a chain (`.invoke`, `.stream`, `.batch`), LangChain needs to **track what’s happening inside**.
That “memory of the execution” is called the **running state**.

It holds things like:

* The current input and output values.
* Intermediate steps in the pipeline.
* Metadata (like tags, IDs, tracing info).
* Config (like max retries, timeouts, callbacks).

Think of it as:
👉 *a diary that records the journey of your data as it flows through the chain.*

---

## 🔹 Why is it useful?

1. **Debugging** → You can trace exactly what happened step by step.
2. **Streaming** → As tokens come in, state helps resume and control the process.
3. **Batching** → Keeps track of multiple items at once.
4. **Custom control** → You can cancel, retry, or log based on state.

---

## 🔹 How it looks in practice

Let’s use a simple example with `stream` (because streaming makes state very visible):

```python
from langchain.schema.runnable import RunnableLambda

# Step: uppercase a string
to_upper = RunnableLambda(lambda x: x.upper())

# Stream the result (character by character, as state evolves)
for chunk in to_upper.stream("hello"):
    print("Chunk:", chunk)
```

**Behind the scenes:**

* The running state stores `"hello"` as input.
* Applies `upper()`.
* Yields `"H"`, then `"E"`, then `"L"`, etc.
* At each yield, state updates: “position = 3, output so far = 'HEL' ”.

---

## 🔹 Advanced: Customizing with `RunnableConfig`

You can pass a config into a chain run:

```python
from langchain.schema.runnable import RunnableConfig

config = RunnableConfig(
    tags=["debug", "experiment1"], 
    metadata={"source": "unit-test"}
)

result = to_upper.invoke("test", config=config)
```

Now the **running state** knows:

* This run has tags `["debug", "experiment1"]`.
* Metadata says it’s from `"unit-test"`.

That info gets attached to every intermediate step → super useful for logging & observability.

---

✅ In summary:

* **Running state** = the internal “progress log” of your chain execution.
* It tracks input/output, metadata, config, and intermediate steps.
* It’s what enables **debugging, streaming, retries, and batch execution** in LCEL.

---

Perfect — let’s design one together.
We’ll keep it **professional-level**, but still short enough so you can follow and re-code it yourself.

---

### Problem Statement

Imagine you’re building a **customer feedback analyzer** for a company.

* Customers leave short comments.
* You want to:

  1. Clean the text (remove noise).
  2. Classify the sentiment (positive, negative, neutral).
  3. Generate a one-line summary of the feedback.
* All of this should happen **in a single chain** using LangChain’s running state.

---

### Code Example

```python
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough
from langchain.schema.runnable.passthrough import RunnableAssign
from langchain.prompts import ChatPromptTemplate
from typing import Dict
import re

# Step 1: Simple cleaner
def clean_text(data: Dict[str, str]) -> Dict[str, str]:
    text = data["input"]
    cleaned = re.sub(r"[^a-zA-Z0-9\s]", "", text).strip()
    return {"cleaned": cleaned}

# Step 2: Rule-based sentiment (for demo)
def classify_sentiment(data: Dict[str, str]) -> Dict[str, str]:
    text = data["cleaned"].lower()
    if any(word in text for word in ["love", "great", "excellent"]):
        return {"sentiment": "positive"}
    elif any(word in text for word in ["bad", "hate", "terrible"]):
        return {"sentiment": "negative"}
    else:
        return {"sentiment": "neutral"}

# Step 3: Prompt for summary generation
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this customer feedback in one line:\n\n{cleaned}"
)

# Step 4: Build chain
feedback_chain = (
    RunnableLambda(clean_text)
    | RunnableAssign({"sentiment": classify_sentiment})
    | RunnableAssign({"summary": summary_prompt | RunnablePassthrough()})
)

# Test input
output = feedback_chain.invoke({"input": "I love the product, but delivery was terrible!!!"})
print(output)
```

---

### What Happens Step by Step

1. **Input:** `"I love the product, but delivery was terrible!!!"`
2. `clean_text` → removes punctuation → `"I love the product but delivery was terrible"`
3. `classify_sentiment` → finds both “love” (positive) and “terrible” (negative) → returns `{"sentiment": "negative"}` (depending on rule priority).
4. `summary_prompt` → generates a short summary like *"Customer likes the product but had issues with delivery."*

---

This gives you a **professional-level pipeline** that’s still short, but uses:

* `RunnableLambda` (custom logic)
* `RunnableAssign` (to add state)
* Prompt templates

---


In [ ]:
from langchain.schema.runnable import RunnableLambda
from langchain.prompts import ChatPromptTemplate
from typing import Dict
import re

# Clean the text (remove noise) using a lambda function with RunnableLambda
clean_text = RunnableLambda(lambda data: {"cleaned": re.sub(r"[^a-zA-Z0-9\s]", "", data["input"]).strip()})

# Sentiment(positive, negative, neutral).
def classify_sentiment(data:Dict[str,str]) -> Dict[str,str]:
    text=data["cleaned"].lower()
    if any(word in text for word in ["love", "great", "excellent"]):
        return {"sentiment":"positive"}
    elif any(word in text for word in ["bad", "hate", "terrible"]):
        return {"sentiment": "negative"}
    else:
        return {"sentiment": "neutral"}
    
#one line summary generation
summary_prompt = ChatPromptTemplate.from_template(
    "summarize this customer feedback in one line: \n\n{cleaned}"
)

#build chain
feedback_chain = (
    clean_text
    | RunnableAssign({"sentiment": classify_sentiment})
    | RunnableAssign({"summary": summary_prompt | RunnablePassthrough()})
)

# Test input
output = feedback_chain.invoke({"input": "I love the product, but delivery was terrible!!!"})
print(output)

{'cleaned': 'I love the product but delivery was terrible', 'sentiment': {'sentiment': 'positive'}, 'summary': ChatPromptValue(messages=[HumanMessage(content='summarize this customer feedback in one line: \n\nI love the product but delivery was terrible', additional_kwargs={}, response_metadata={})])}
